c:\Users\Lenovo\OneDrive\Desktop\Lanchain_k\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18616\2634015303.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


ModuleNotFoundError: No module named 'langchain.chains'

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['Groq_API_KEY']=os.getenv("Groq_API_KEY")
from langchain_groq import ChatGroq
model =ChatGroq(model="groq/compound")
model

c:\Users\Lenovo\OneDrive\Desktop\Lanchain_k\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'name': 'Compound', 'release_date': '2025-09-04', 'last_updated': '2025-09-04', 'open_weights': False, 'max_input_tokens': 131072, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CDBA7C6320>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CDBA7C6A10>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Lenovo\OneDrive\Desktop\Lanchain_k\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2673.86it/s]


In [43]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableSequence

from langchain_core.output_parsers import StrOutputParser


In [11]:
# Target LangChain blog URL
url = "https://www.langchain.com/blog/how-to-build-a-custom-agent-harness"

# Initialize the loader
loader = WebBaseLoader(url)

# Load the data into a LangChain Document format
docs = loader.load()

# Extract and view the content
blog_content = docs[0].page_content
print(f"Loaded {len(blog_content)} characters.")
print("\n--- Snippet of Scraped Blog ---\n")
print(blog_content[:500])

Loaded 9631 characters.

--- Snippet of Scraped Blog ---

How to Build a Custom Agent Harness













































Products

LangSmith Platform

EngineImprove agents autonomouslyObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionFleetAgents for the whole companySandboxesRun agent-generated code safelyOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick start agents with any model providerlanggraphBuild r


In [25]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=text_splitter.split_documents(docs)
chromadb=Chroma.from_documents(chunks, embeddings)
retriever=chromadb.as_retriever()

In [44]:
#  CORRECT PROMPT CONFIGURATION
system_template = """You are a helpful assistant for answering questions about the content of the blog post. 

Use ONLY the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Do not make up information.

Retrieved Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}")
])

# Helper function to format retrieved Document objects into a single clean string
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. OPTIMIZE THE RETRIEVER: Keep it lean (top 2 chunks)
retriever = chromadb.as_retriever(search_kwargs={"k": 2})

# 2. BRIDGING FUNCTION: Clean and join the document text
def format_docs(docs_list):
    return "\n\n".join(doc.page_content for doc in docs_list)

# 3. LOCAL TEXT SAFE-TRIMMER: Cuts down text length locally if it's too long
def limit_context_tokens(text: str) -> str:
    # A quick, offline rule of thumb: 1 word ≈ 1.3 tokens. 
    # This keeps the text well under Groq's 413 payload limit.
    words = text.split()
    if len(words) > 300: # Adjust to roughly 300-400 words max
        return " ".join(words[:300])
    return text

# 4. CONFIGURE THE RAG CHAIN CORRECTLY
rag_chain = (
    {
        # Extract question -> Retrieve -> Format to text -> Trim text locally
        "context": itemgetter("question") | retriever | format_docs | limit_context_tokens, 
        "question": itemgetter("question")
    }
    | prompt
    | model
    | StrOutputParser()
)




In [39]:
ans=rag_chain.invoke({"question": "What is the block topic is about in summary?"})

In [40]:
print(ans)

The block outlines the **middleware capabilities** that enhance an AI agent’s operation:

- **Preventing context overflow** – using tools like SummarizationMiddleware and ContextEditingMiddleware to keep long sessions from exceeding the context window.  
- **Accessing and updating memory** – loading relevant knowledge at startup and writing it back afterward with FilesystemMiddleware, MemoryMiddleware, and SkillsMiddleware so the agent can learn from real usage.  
- **Taking actions in an environment** – expanding the agent’s toolset (ShellToolMiddleware, FilesystemMiddleware, CodeInterpreterMiddleware) to interact with a filesystem and execution environment, enabling more creative and token‑efficient solutions.  
- **Delegating tasks** – employing sub‑agents (SubAgentMiddleware, AsyncSubAgentMiddleware) and a TodoListMiddleware to handle complex sub‑tasks while keeping context windows clean.  

In short, the topic describes how different middleware components help an AI agent manage c

*ADDING CHAT HISTORY*

In [55]:
from operator import itemgetter
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# 1. SESSION MEMORY MANAGEMENT
store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 2. FIREWALL-PROOF RETRIEVAL LOGIC
# This intercepts documents immediately and converts them to short text snippets
def safe_retrieve_text(inputs):
    question_str = inputs["question"]
    # Request only 2 chunks to minimize data size
    retriever = chromadb.as_retriever(search_kwargs={"k": 2})
    raw_docs = retriever.invoke(question_str)
    
    # Extract raw text content immediately
    full_text = "\n\n".join(doc.page_content for doc in raw_docs)
    
    # Strictly limit the text to the first 150 words
    words = full_text.split()
    if len(words) > 150:
        return " ".join(words[:150])
    return full_text

# Convert this into a clean, standalone Runnable element
context_runnable = RunnableLambda(safe_retrieve_text)

# 3. CHAT HISTORY SLIDING WINDOW
def limit_chat_history(messages_list):
    # Keep only the last 2 messages (1 Human turn, 1 AI turn)
    if len(messages_list) > 2:
        return messages_list[-2:]
    return messages_list

# 4. STRUCTURED SYSTEM PROMPT
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant. Answer using ONLY this context:\n\n{context}"),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{question}")                      
])

# 5. CORE LCEL CHAIN LINKING
base_chain = (
    {
        "context": context_runnable,
        "question": itemgetter("question"),
        # Trim history down before sending to the template
        "history": itemgetter("history") | RunnableLambda(limit_chat_history)
    }
    | prompt
    | model
    | StrOutputParser()
)

# 6. CONNECT HISTORY AUTOMATION
conversational_rag_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)


c:\Users\Lenovo\OneDrive\Desktop\Lanchain_k\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [59]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# 1. SIMPLE RETRIEVER snip (keeps payloads incredibly light)
def simple_scraped_context(inputs):
    question_text = inputs["question"]
    # Retrieve only the single best matching document chunk
    retriever = chromadb.as_retriever(search_kwargs={"k": 1})
    docs = retriever.invoke(question_text)
    
    # Extract text content and keep it under 100 words max
    text = " ".join([doc.page_content for doc in docs])
    return " ".join(text.split()[:100])

context_runnable = RunnableLambda(simple_scraped_context)


# 2. STANDARD CLEAN PROMPT (Requires raw strings to prevent 400 alternating errors)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant. Answer using ONLY this context:\n\n{context}"),
    # Injected as raw text to keep it completely metadata-free
    ("system", "Previous Chat Context:\n{history_string}"),
    ("human", "{question}")
])


# 3. DIRECT LCEL PIPELINE (No bloated wrappers)
clean_rag_chain = (
    {
        "context": context_runnable,
        "question": itemgetter("question"),
        "history_string": itemgetter("history_string")
    }
    | prompt
    | model
    | StrOutputParser()
)


In [60]:
# Create a light text tracker string variable instead of a heavy object store
history_log = ""

# --- TURN 1 ---
question1 = "What is the Harness capabilities?"
response1 = clean_rag_chain.invoke({
    "question": question1,
    "history_string": history_log if history_log else "No prior history."
})
print("Response 1:\n", response1)

# Manually update our string tracker with a compact log snippet
history_log = f"User asked: {question1}\nAI replied: {response1[:60]}..."

print("\n" + "="*40 + "\n")

# --- TURN 2 --- (Will execute perfectly with zero hidden metadata payload)
question2 = "What is harness components?"
response2 = clean_rag_chain.invoke({
    "question": question2,
    "history_string": history_log
})
print("Response 2:\n", response2)


APIStatusError: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}